# 7. RGB Counting Baseline Colab

This notebook starts the next research branch after the pose-only baselines. It builds a small frozen RGB baseline on a controlled subset of exercises and compares it directly against the strongest pose-only `6B` runs.


## Reason, Approach, Result Interpretation

**Reason**
- The pose-only branch produced meaningful baselines, but absolute errors remained high.
- `6C` and `6D` showed that more small pose-side tweaks are unlikely to be the main answer.
- The next question is whether a stronger visual representation helps where pose-only is strongest, weakest, and most ambiguous.

**Approach**
- Keep the task the same: per-exercise repetition counting.
- Work on a small controlled subset: `squat`, `pull_up`, and `push_up`.
- Extract frozen RGB frame features with a pretrained ResNet18 backbone.
- Train per-exercise TCN baselines on those RGB feature sequences.
- Compare the RGB results against the best pose-only `6B` runs and the trivial train-split baseline.

**Result interpretation**
- If RGB beats pose-only on a hard exercise like `push_up`, that suggests pose is missing useful context.
- If pose still beats RGB on `squat` or `pull_up`, pose remains competitive for those classes.
- The goal is not to replace the whole pipeline immediately, but to decide whether the research should stay pose-first or move toward RGB or multimodal counting.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup

**Why this section exists**
- The RGB branch needs its own extraction and training scripts, plus the existing baseline-comparison script.

**Approach**
- Mount Drive.
- Sync the current repo copies of the RGB extraction script, RGB trainer, and baseline-comparison script into the Drive project.
- Resolve the full metadata index and raw video directory.

**How to interpret the result**
- If the printed paths exist, the environment is ready.
- If the video directory or metadata index is missing, fix that before running the RGB stage.


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

SCALAR_TRAINER_REL = Path('artifacts/3_Modeling/train_pose_count_tcn.py')
RGB_EXTRACT_REL = Path('artifacts/3_Modeling/extract_rgb_frame_features.py')
RGB_TRAIN_REL = Path('artifacts/3_Modeling/train_rgb_count_tcn.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')

def sync_drive_file(rel: Path) -> None:
    src = CODE_ROOT / rel
    dst = DRIVE_PROJECT_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    if not src.exists() and not dst.exists():
        print(f'[sync] missing both copies: {rel}')
        return
    if not src.exists():
        print(f'[sync] keeping Drive copy (no /content source): {rel}')
        return
    if not dst.exists():
        shutil.copy2(src, dst)
        print(f'[sync] copied /content -> Drive (Drive missing): {rel}')
        return

    src_mtime = src.stat().st_mtime
    dst_mtime = dst.stat().st_mtime
    if src_mtime > dst_mtime + 1.0:
        shutil.copy2(src, dst)
        print(f'[sync] copied newer /content -> Drive: {rel}')
    else:
        print(f'[sync] keeping Drive copy (newer or equal): {rel}')

for rel in [SCALAR_TRAINER_REL, RGB_EXTRACT_REL, RGB_TRAIN_REL, COMPARE_REL]:
    sync_drive_file(rel)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
VIDEO_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/video'
POSE_INDEX = ANNOTATION_DIR / 'pose_feature_index.csv'
RGB_INDEX = ANNOTATION_DIR / 'rgb_feature_index_selected.csv'
RGB_SUMMARY = ANNOTATION_DIR / 'rgb_feature_summary_selected.csv'
RGB_FEATURE_DIR = ANNOTATION_DIR / 'rgb_resnet18_features'

print('SCALAR_TRAINER =', DRIVE_PROJECT_ROOT / SCALAR_TRAINER_REL)
print('RGB extractor logging flags present =', '--log-every' in (DRIVE_PROJECT_ROOT / RGB_EXTRACT_REL).read_text())
print('POSE_INDEX =', POSE_INDEX)
print('POSE_INDEX exists =', POSE_INDEX.exists())
print('VIDEO_DIR =', VIDEO_DIR)
print('VIDEO_DIR exists =', VIDEO_DIR.exists())
print('RGB_FEATURE_DIR =', RGB_FEATURE_DIR)


## Controlled Exercise Subset

**Why this section exists**
- The first RGB comparison should stay small and interpretable.
- We want one strong pose control (`squat`), one promising pose class (`pull_up`), and one unresolved class (`push_up`).

**Approach**
- Limit the RGB branch to `squat`, `pull_up`, and `push_up`.
- Reuse the best `seq_len` from `6B` for fair comparison.

**How to interpret the result**
- If RGB helps only `push_up`, it suggests pose is mainly missing context on the hardest class.
- If RGB wins broadly, the research should lean more strongly toward visual or multimodal representations.


In [ ]:
import pandas as pd

TARGET_EXERCISES = ['squat', 'pull_up', 'push_up']
POSE_SEQ_LENS = {
    'squat': 256,
    'pull_up': 192,
    'push_up': 128,
}

meta_df = pd.read_csv(POSE_INDEX)
subset_counts = meta_df[meta_df['type'].isin(TARGET_EXERCISES)].groupby(['type', 'split']).size().unstack(fill_value=0).sort_index()
display(subset_counts)
print('POSE_SEQ_LENS =', POSE_SEQ_LENS)


## RGB Feature Extraction

**Why this section exists**
- The RGB trainer works on frozen frame-feature sequences rather than raw videos.
- This keeps the RGB branch comparable to the pose branch and makes repeated experiments practical.

**Approach**
- Read the existing metadata index.
- Filter to `squat`, `pull_up`, and `push_up`.
- Resolve each raw video by name and extract frozen ResNet18 frame embeddings.
- Write a new `rgb_feature_index_selected.csv` plus an extraction summary.

**How to interpret the result**
- `ok` rows mean the RGB feature sequence was written successfully.
- If many rows fail, the issue is data resolution or video access rather than the RGB TCN itself.


In [ ]:
import subprocess

extract_cmd = [
    'python', '-u', str(DRIVE_PROJECT_ROOT / RGB_EXTRACT_REL),
    '--index-csv', str(POSE_INDEX),
    '--video-dir', str(VIDEO_DIR),
    '--feature-dir', str(RGB_FEATURE_DIR),
    '--output-index-csv', str(RGB_INDEX),
    '--output-summary-csv', str(RGB_SUMMARY),
    '--max-frames', '256',
    '--batch-size', '32',
    '--device', 'cuda',
    '--overwrite',
    '--log-every', '1',
    '--save-progress-every', '0',
]
for exercise in TARGET_EXERCISES:
    extract_cmd.extend(['--exercise', exercise])

print('Running:', ' '.join(extract_cmd))
subprocess.run(extract_cmd, check=True)


In [ ]:
rgb_summary_df = pd.read_csv(RGB_SUMMARY)
display(rgb_summary_df['status'].value_counts())
display(rgb_summary_df.groupby(['type', 'status']).size().unstack(fill_value=0).sort_index())
display(rgb_summary_df.groupby('type')[['frames_total', 'frames_used', 'feature_dim']].mean().round(2))


## RGB TCN Training

**Why this section exists**
- This is the first learned RGB counting baseline.

**Approach**
- Train one RGB TCN per exercise.
- Keep the training recipe aligned with Stage 6 as much as practical.
- Reuse the best pose `seq_len` per exercise for a cleaner representation comparison.

**How to interpret the result**
- A completed run means the RGB branch is operational.
- The next sections decide whether it actually beats pose and whether it adds value over a trivial baseline.


In [ ]:
import subprocess
import pandas as pd

RGB_RUNS = [
    {
        'exercise': 'squat',
        'seq_len': 256,
        'run_name': 'rgb_count_tcn_squat_seq256',
        'pose_run': 'pose_count_tcn_squat_seq256',
        'pose_best_run': 'squat_tcn_l1_channels96',
    },
    {
        'exercise': 'pull_up',
        'seq_len': 192,
        'run_name': 'rgb_count_tcn_pull_up_seq192',
        'pose_run': 'pose_count_tcn_pull_up_seq192',
    },
    {
        'exercise': 'push_up',
        'seq_len': 128,
        'run_name': 'rgb_count_tcn_push_up_seq128',
        'pose_run': 'pose_count_tcn_push_up_seq128',
    },
]

training_failures = []
for cfg in RGB_RUNS:
    cmd = [
        'python', str(DRIVE_PROJECT_ROOT / RGB_TRAIN_REL),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--index-csv', str(RGB_INDEX),
        '--run-name', cfg['run_name'],
        '--exercise', cfg['exercise'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', '80',
        '--batch-size', '16',
        '--lr', '0.001',
        '--weight-decay', '0.0001',
        '--channels', '96',
        '--kernel-size', '3',
        '--num-blocks', '4',
        '--dropout', '0.2',
        '--patience', '15',
        '--loss', 'l1',
        '--eval-transform', 'raw',
        '--selection-metric', 'mae',
        '--sampler', 'balanced_count',
        '--time-warp-range', '0.12',
        '--feature-noise-std', '0.02',
        '--frame-dropout-prob', '0.03',
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        training_failures.append({
            'exercise': cfg['exercise'],
            'run_name': cfg['run_name'],
            'returncode': exc.returncode,
        })
        print(f"FAILED: {cfg['exercise']} (returncode={exc.returncode})")

if training_failures:
    display(pd.DataFrame(training_failures))
else:
    print('All RGB runs completed.')


## Pose vs RGB Metric Review

**Why this section exists**
- The main research question is whether RGB improves counting beyond the strongest pose baseline on the same exercises.

**Approach**
- Load `metrics_summary.json` for the best pose `6B` run and the new RGB run.
- Compare `valid_mae`, `valid_rmse`, and `valid_within_1` side by side.

**How to interpret the result**
- Lower `MAE` and higher `Within-1` in the RGB row indicate that RGB is helping.
- Mixed results suggest the project may need a multimodal rather than a pure RGB replacement.


In [ ]:
import json
import pandas as pd

rows = []
for cfg in RGB_RUNS:
    variants = [('pose_6B', cfg['pose_run']), ('rgb_tcn', cfg['run_name'])]
    if cfg.get('pose_best_run'):
        variants.insert(1, ('pose_best_squat', cfg['pose_best_run']))
    for variant, run_name in variants:
        metrics_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / run_name / 'metrics_summary.json'
        if not metrics_path.exists():
            continue
        with open(metrics_path, 'r', encoding='utf-8') as f:
            metrics = json.load(f)
        rows.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'best_epoch': metrics.get('best_epoch'),
            'valid_mae': metrics['valid_metrics']['mae'],
            'valid_rmse': metrics['valid_metrics']['rmse'],
            'valid_within_1': metrics['valid_metrics']['within_1'],
        })

compare_df = pd.DataFrame(rows)
display(compare_df.sort_values(['exercise', 'variant']))

pivot_df = compare_df.pivot(index=['exercise', 'seq_len'], columns='variant', values=['valid_mae', 'valid_within_1'])
pivot_df.columns = ['_'.join(col).strip() for col in pivot_df.columns.values]
pivot_df = pivot_df.reset_index()
if 'valid_mae_pose_6B' in pivot_df.columns and 'valid_mae_rgb_tcn' in pivot_df.columns:
    pivot_df['delta_mae_rgb_minus_pose6B'] = pivot_df['valid_mae_rgb_tcn'] - pivot_df['valid_mae_pose_6B']
if 'valid_within_1_pose_6B' in pivot_df.columns and 'valid_within_1_rgb_tcn' in pivot_df.columns:
    pivot_df['delta_within_1_rgb_minus_pose6B'] = pivot_df['valid_within_1_rgb_tcn'] - pivot_df['valid_within_1_pose_6B']
if 'valid_mae_pose_best_squat' in pivot_df.columns and 'valid_mae_rgb_tcn' in pivot_df.columns:
    pivot_df['delta_mae_rgb_minus_best_squat'] = pivot_df['valid_mae_rgb_tcn'] - pivot_df['valid_mae_pose_best_squat']
if 'valid_within_1_pose_best_squat' in pivot_df.columns and 'valid_within_1_rgb_tcn' in pivot_df.columns:
    pivot_df['delta_within_1_rgb_minus_best_squat'] = pivot_df['valid_within_1_rgb_tcn'] - pivot_df['valid_within_1_pose_best_squat']
display(pivot_df.sort_values('exercise'))


## RGB Baseline Comparison Review

**Why this section exists**
- A better pose-vs-RGB comparison still needs context: does the RGB model actually beat the trivial train-split count baseline?

**Approach**
- Run the existing baseline-comparison script on the RGB runs.
- Compare the RGB rows to both the trivial baseline and the best pose `6B` row.

**How to interpret the result**
- If RGB beats the trivial baseline and beats pose on the same exercise, it is a strong next-stage direction.
- If RGB does not even beat the trivial baseline, the representation change alone is not enough.


In [ ]:
import subprocess
import json
import pandas as pd

comparison_failures = []
for cfg in RGB_RUNS:
    run_dir = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['run_name']
    pred_path = run_dir / 'predictions.csv'
    if not pred_path.exists():
        continue
    summary_path = run_dir / 'baseline_comparison_summary.json'
    if summary_path.exists():
        continue
    cmd = [
        'python', str(DRIVE_PROJECT_ROOT / COMPARE_REL),
        '--index-csv', str(RGB_INDEX),
        '--predictions-csv', str(pred_path),
        '--exercise', cfg['exercise'],
    ]
    print('\nComparing:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        comparison_failures.append({
            'exercise': cfg['exercise'],
            'run_name': cfg['run_name'],
            'returncode': exc.returncode,
        })

rows = []
for cfg in RGB_RUNS:
    for variant, run_name, index_path in [
        ('pose_6B', cfg['pose_run'], ANNOTATION_DIR / 'pose_sequence_index.csv'),
        ('rgb_tcn', cfg['run_name'], RGB_INDEX),
    ]:
        summary_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / run_name / 'baseline_comparison_summary.json'
        if not summary_path.exists():
            continue
        with open(summary_path, 'r', encoding='utf-8') as f:
            summary = json.load(f)
        rows.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'model_mae': summary['model_metrics']['mae'],
            'baseline_mae': summary['baseline_metrics']['mae'],
            'delta_mae_vs_trivial': summary['delta_vs_baseline']['mae'],
            'model_within_1': summary['model_metrics']['within_1'],
            'baseline_within_1': summary['baseline_metrics']['within_1'],
            'delta_within_1_vs_trivial': summary['delta_vs_baseline']['within_1'],
            'model_beats_baseline_rows': summary['row_level']['model_beats_baseline'],
            'valid_rows': summary['row_level']['valid_rows'],
        })

baseline_df = pd.DataFrame(rows)
display(baseline_df.sort_values(['exercise', 'variant']))

pivot_df = baseline_df.pivot(index=['exercise', 'seq_len'], columns='variant', values=['model_mae', 'model_within_1', 'delta_mae_vs_trivial', 'delta_within_1_vs_trivial'])
pivot_df.columns = ['_'.join(col).strip() for col in pivot_df.columns.values]
pivot_df = pivot_df.reset_index()
if 'model_mae_pose_6B' in pivot_df.columns and 'model_mae_rgb_tcn' in pivot_df.columns:
    pivot_df['delta_mae_rgb_minus_pose6B'] = pivot_df['model_mae_rgb_tcn'] - pivot_df['model_mae_pose_6B']
if 'model_within_1_pose_6B' in pivot_df.columns and 'model_within_1_rgb_tcn' in pivot_df.columns:
    pivot_df['delta_within_1_rgb_minus_pose6B'] = pivot_df['model_within_1_rgb_tcn'] - pivot_df['model_within_1_pose_6B']
display(pivot_df.sort_values('exercise'))

if comparison_failures:
    display(pd.DataFrame(comparison_failures))
